In [1]:
!date

Mon Sep 14 07:49:05 PDT 2026


In [2]:
import pandas as pd
import numpy as np
import glob
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

projdir = '/u/project/cluo/terencew/claude/project_ideas/pool_design'
sample = '20220928-IGVF-D0'

In [3]:
### discover pools with at least one modality demuxlet'd so far
pools = sorted(set(p.split('/ambisim/')[1].split('/')[0] for p in
    glob.glob(f'{projdir}/ambisim/*/demux/demuxlet/*/{sample}.best')))
len(pools)

88

In [4]:
### report readiness against the full pool_experiments.txt design (132 pools). This notebook
### is rerun incrementally as more of the ambisim array finishes, so always show what's still
### missing/partial rather than silently just working on whatever happens to be on disk.
all_pools = [l.strip() for l in open(f'{projdir}/ambisim/txt/pool_experiments.txt')]
best_files = glob.glob(f'{projdir}/ambisim/*/demux/demuxlet/*/{sample}.best')
have = {}
for p in best_files:
    pool = p.split('/ambisim/')[1].split('/')[0]
    modality = p.split('/demuxlet/')[1].split('/')[0]
    have.setdefault(pool, set()).add(modality)

ready = [p for p in all_pools if have.get(p, set()) == {'gex', 'atac'}]
partial = {p: sorted(have[p]) for p in all_pools if p in have and have[p] != {'gex', 'atac'}}
not_started = [p for p in all_pools if p not in have]

print(f'{len(ready)}/{len(all_pools)} pools ready (both modalities scored)')
print(f'{len(partial)} partial (only one modality -- usually the demuxlet call OOM-killed '
      f'for the larger-pileup modality; rerun that pool\'s call with more memory):')
for p, mods in partial.items():
    print(f'    {p}: has {mods}')
print(f'{len(not_started)} not started yet (no .best for either modality)')

88/132 pools ready (both modalities scored)
0 partial (only one modality -- usually the demuxlet call OOM-killed for the larger-pileup modality; rerun that pool's call with more memory):
44 not started yet (no .best for either modality)


In [5]:
### join demuxlet .best calls against ambisim's known donor-of-origin / ambient RNA ground truth
def load_pool_modality(pool, modality):
    best_path = f'{projdir}/ambisim/{pool}/demux/demuxlet/{modality}/{sample}.best'
    if not os.path.exists(best_path):
        return None
    best = pd.read_csv(best_path, sep='\t')
    best['barcode'] = best['BARCODE'].str.replace('-1', '', regex=False)
    truth = pd.read_csv(f'{projdir}/ambisim/{pool}/{sample}/drop_data_rand.txt', sep='\t',
                         dtype={'sam': str, 'ct': str})
    # both GEX and ATAC demuxlet were run against the same RNA-space barcode list
    # (A02b_demuxlet_call.sh passes one $BARCODES to both calls), so ATAC .best
    # BARCODE is already translated into RNA_BC space, not raw ATAC_BC
    truth = truth.rename(columns={'RNA_BC': 'barcode'})
    df = truth.merge(best[['barcode', 'DROPLET.TYPE', 'SNG.BEST.GUESS', 'DIFF.LLK.BEST.NEXT']],
                      on='barcode', how='inner')
    if modality == 'gex':
        df['ambient_frac'] = df['rna_nr_a'] / (df['rna_nr_a'] + df['rna_nr_c'])
    else:
        df['ambient_frac'] = df['atac_nr_a'] / (df['atac_nr_a'] + df['atac_nr_c'])
    df['called_donor'] = df['SNG.BEST.GUESS'].str.split(',').str[0]
    df['is_true_singlet'] = df['n'] == 1
    df['called_singlet'] = df['DROPLET.TYPE'] == 'SNG'
    df['correct'] = df['is_true_singlet'] & df['called_singlet'] & (df['called_donor'] == df['sam'])
    df['ll_gap'] = df['DIFF.LLK.BEST.NEXT']
    universe, strategy, rep = pool.split('__')
    df['pool'] = pool
    df['universe'] = universe
    df['strategy'] = strategy
    df['rep'] = int(rep.replace('rep', ''))
    df['modality'] = modality
    df = df.rename(columns={'sam': 'true_donor'})
    return df[['pool', 'universe', 'strategy', 'rep', 'modality', 'barcode',
               'is_true_singlet', 'called_singlet', 'correct', 'ambient_frac', 'll_gap',
               'called_donor', 'true_donor']]

def _load_job(job):
    return load_pool_modality(*job)

In [6]:
### load every pool x modality in parallel
jobs = [(pool, modality) for pool in pools for modality in ['gex', 'atac']]
with ProcessPoolExecutor(max_workers=10) as ex:
    results = list(tqdm(ex.map(_load_job, jobs), total=len(jobs)))
scored = pd.concat([r for r in results if r is not None], ignore_index=True)

100%|██████████| 176/176 [00:22<00:00,  7.67it/s]


In [7]:
scored.head()

,pool,universe,strategy,rep,modality,barcode,is_true_singlet,called_singlet,correct,ambient_frac,ll_gap,called_donor,true_donor
0,AFR_only__adversarial_mindist__rep1,AFR_only,adversarial_mindist,1,gex,AAACAGCCAAACAACA,True,True,True,0.098320,15.04,HG03461,HG03461
1,AFR_only__adversarial_mindist__rep1,AFR_only,adversarial_mindist,1,gex,AAACAGCCAAACATAG,True,True,True,0.156914,10.97,NA18924,NA18924
2,AFR_only__adversarial_mindist__rep1,AFR_only,adversarial_mindist,1,gex,AAACAGCCAAACCCTA,True,False,False,0.391450,0.00,HG03520,HG03520
3,AFR_only__adversarial_mindist__rep1,AFR_only,adversarial_mindist,1,gex,AAACAGCCAAACCTAT,True,False,False,0.314493,0.00,NA18933,NA18933
4,AFR_only__adversarial_mindist__rep1,AFR_only,adversarial_mindist,1,gex,AAACAGCCAAACCTTG,True,True,True,0.184638,20.95,NA19150,NA19150


In [8]:
scored.shape

(1583824, 13)

In [9]:
### only keep pools where both gex and atac demuxlet finished (some are still mid-run)
complete_pools = scored.groupby('pool')['modality'].nunique()
complete_pools = complete_pools[complete_pools == 2].index
print(len(complete_pools), 'of', len(pools), 'pools have both modalities scored')
scored = scored[scored['pool'].isin(complete_pools)].reset_index(drop=True)

88 of 88 pools have both modalities scored


In [10]:
scored.shape

(1583824, 13)

In [11]:
### write checkpoint for downstream pool-level / droplet-level notebooks
outdir = f'{projdir}/csv/ambisim'
os.makedirs(outdir, exist_ok=True)
scored.to_csv(f'{outdir}/droplet_scores.csv', sep='\t', index=False)

In [12]:
!date

Mon Sep 14 07:49:37 PDT 2026
